# Ultimate Production Ensemble Pipeline

This notebook implements the definitive, production-ready architecture for the Speech Emotion Recognition (SER) project. It bridges the gap between academic evaluation and real-world software engineering.

### Architectural Masterpieces:
1. **Full Acoustic Integrity**: Dynamically ingests all **48 features** without arbitrary pruning, ensuring maximum data resolution.
2. **Soft-Voting Ensemble**: Blends **XGBoost**, **LightGBM**, and **CatBoost** using probabilities, compensating for the weaknesses of any single algorithm.
3. **Out-of-Fold (OOF) Weight Optimization**: Uses 3-Fold Stratified Cross-Validation strictly on the training set to mathematically prove the optimal blending weights (e.g., 40% XGB, 42% LGBM, 18% CB) without data leakage.
4. **Deployment Serialization**: Automatically exports the trained models, the data scaler, the label encoder, and the optimized weights directly to disk for instant loading in a live FastAPI/Flask backend.

In [ ]:
# ============================================================
# 1. Environment Setup & Library Imports
# ============================================================
import os
import json
import numpy as np
import pandas as pd
import joblib
import warnings
import torch

# Safely import boosting libraries
try: import xgboost as xgb
except ImportError: pass
try: import lightgbm as lgb
except ImportError: pass
try: from catboost import CatBoostClassifier
except ImportError: pass

from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
CUDA_AVAILABLE = torch.cuda.is_available()
print("CUDA Available (PyTorch):", CUDA_AVAILABLE)

CUDA Available (PyTorch): False


### 2. Data Ingestion & Preprocessing
We dynamically locate `all_emotions.csv`. We extract all 48 acoustic features and apply basic median imputation to protect the models from crashing on `NaN` or `inf` values caused by audio extraction anomalies.

In [ ]:
# Resolve Paths
data_path = os.path.join('dataset', 'all_emotions.csv')
if not os.path.exists(data_path):
    data_path = 'all_emotions.csv' # Fallback

print(f"Loading dataset from: {data_path}")
df = pd.read_csv(data_path)

# Detect Target Column
target_col = "label" if "label" in df.columns else "Emotion"
if target_col not in df.columns:
    target_col = df.columns[-1]

# Clean Missing Labels
df_cleaned = df.dropna(subset=[target_col]).copy()
df_cleaned = df_cleaned[df_cleaned[target_col].astype(str).str.strip().str.lower() != "nan"]

# Feature Extraction & Imputation
FEATURE_COLS = [col for col in df_cleaned.columns if col not in [target_col]]
print(f"Number of features selected dynamically: {len(FEATURE_COLS)}")

for col in FEATURE_COLS:
    s = pd.to_numeric(df_cleaned[col], errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    df_cleaned[col] = s.fillna(s.median() if not pd.isna(s.median()) else 0.0)

X = df_cleaned[FEATURE_COLS].values
y_label = df_cleaned[target_col].astype(str).str.strip().values

# Encode Labels
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y_label)

Loading dataset from: dataset\all_emotions.csv
Number of features selected dynamically: 48


### 3. Pipeline Splitting & Scaling
We implement an 80/20 Stratified Train-Test Split. The `StandardScaler` is fitted *strictly* on the training set to prevent data leakage into the test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=RANDOM_STATE, stratify=y_encoded
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Training Set: {X_train_scaled.shape[0]} samples")
print(f"Testing Set: {X_test_scaled.shape[0]} samples")

Training Set: 43588 samples
Testing Set: 10897 samples


### 4. Model Instantiation
We initialize the three heavyweights: XGBoost, LightGBM, and CatBoost. 
The script attempts to load hyper-parameters from `best_params.json` if it exists (from previous GridSearch/Optuna runs). If not, it falls back to highly-optimized default baselines.

In [ ]:
best_params = {}
if os.path.isfile("best_params.json"):
    with open("best_params.json", "r") as f:
        best_params = json.load(f)
    print("Loaded best_params.json successfully.")

xgb_params = best_params.get("xgboost", {
    "n_estimators": 468, "max_depth": 10, "learning_rate": 0.175, 
    "subsample": 0.969, "colsample_bytree": 0.772, "gamma": 1e-08
})
xgb_model = xgb.XGBClassifier(**xgb_params, random_state=RANDOM_STATE, n_jobs=-1, eval_metric="mlogloss", objective="multi:softprob", num_class=len(encoder.classes_), device="cuda" if CUDA_AVAILABLE else "cpu")

lgb_params = best_params.get("lightgbm", {
    "n_estimators": 499, "max_depth": 11, "num_leaves": 67, 
    "learning_rate": 0.246, "subsample": 0.690, "colsample_bytree": 0.755
})
lgb_model = lgb.LGBMClassifier(**lgb_params, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1, objective="multiclass", num_class=len(encoder.classes_), device="gpu" if CUDA_AVAILABLE else "cpu")

cb_params = best_params.get("catboost", {
    "iterations": 500, "depth": 8, "learning_rate": 0.15, "l2_leaf_reg": 3.0
})
cb_model = CatBoostClassifier(**cb_params, loss_function="MultiClass", random_seed=RANDOM_STATE, thread_count=-1, verbose=False, task_type="GPU" if CUDA_AVAILABLE else "CPU")
print("Models instantiated.")

Loaded best_params.json successfully.
Models instantiated.


### 5. Out-Of-Fold (OOF) Weight Optimization
Instead of guessing how much to trust each model, we mathematically prove it.
We split the training data into 3 folds. We train on 2 folds and predict the 3rd. By comparing the predictions against the known truth, we run a grid search to find the exact percentage weighting (e.g., `0.4*XGB + 0.5*LGB + 0.1*CB`) that maximizes the F1-Score.

In [ ]:
print("Training Out-Of-Fold models for weight optimization...")
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

xgb_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))
lgb_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))
cb_oof = np.zeros((len(X_train_scaled), len(encoder.classes_)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
    print(f"  Processing Fold {fold + 1}...")
    X_tr, y_tr = X_train_scaled[train_idx], y_train[train_idx]
    X_va, y_va = X_train_scaled[val_idx], y_train[val_idx]

    # Clone models for clean training on this fold
    xgb_m = xgb.XGBClassifier(**xgb_model.get_params())
    lgb_m = lgb.LGBMClassifier(**lgb_model.get_params())
    cb_m = CatBoostClassifier(**cb_model.get_params())

    xgb_m.fit(X_tr, y_tr)
    lgb_m.fit(X_tr, y_tr)
    cb_m.fit(X_tr, y_tr)

    xgb_oof[val_idx] = xgb_m.predict_proba(X_va)
    lgb_oof[val_idx] = lgb_m.predict_proba(X_va)
    cb_oof[val_idx] = cb_m.predict_proba(X_va)

# Grid Search for Best Weights
best_f1 = 0.0
best_weights = np.array([1.0, 1.0, 1.0]) / 3.0

for w_xgb in np.linspace(0, 1, 11):
    for w_lgb in np.linspace(0, 1 - w_xgb, 11):
        w_cb = 1.0 - w_xgb - w_lgb
        if w_cb < 0: continue
            
        val_proba = (w_xgb * xgb_oof + w_lgb * lgb_oof + w_cb * cb_oof)
        preds = np.argmax(val_proba, axis=1)
        score = f1_score(y_train, preds, average="weighted")
        
        if score > best_f1:
            best_f1 = score
            best_weights = np.array([w_xgb, w_lgb, w_cb])

print(f"\nOptimized Blending Weights: XGBoost={best_weights[0]:.2f}, LightGBM={best_weights[1]:.2f}, CatBoost={best_weights[2]:.2f}")

Training Out-Of-Fold models for weight optimization...
  Processing Fold 1...


KeyboardInterrupt: 

### 6. Final Training & Blind Test Evaluation
Now that we know exactly how to weight the models, we fit them one final time on the **entire 80% training set** to maximize their learning.
We then generate raw probability predictions on the unseen 20% Test Set, apply our custom weights, and generate the ultimate classification report.

In [ ]:
print("\nFitting final models on full scaled training set...")
xgb_model.fit(X_train_scaled, y_train)
lgb_model.fit(X_train_scaled, y_train)
cb_model.fit(X_train_scaled, y_train)

xgb_proba = xgb_model.predict_proba(X_test_scaled)
lgb_proba = lgb_model.predict_proba(X_test_scaled)
cb_proba = cb_model.predict_proba(X_test_scaled)

ensemble_proba = (best_weights[0] * xgb_proba + best_weights[1] * lgb_proba + best_weights[2] * cb_proba)
ensemble_pred = np.argmax(ensemble_proba, axis=1)

print("\n=================== ULTIMATE ENSEMBLE TEST REPORT ===================")
print(classification_report(y_test, ensemble_pred, target_names=encoder.classes_, digits=4))

### 7. Hard-Save Production Artifacts to Disk
**This is the crucial step for production.** We serialize the 3 trained models, the scaler (to maintain mathematical formatting), the label encoder (to translate predictions back to strings), and the optimal weights. 

Your live backend (FastAPI/Flask/Django) will simply `joblib.load()` these files on startup and run inference in milliseconds without ever needing to see the original CSV data.

In [ ]:
print("\nSerializing Production Artifacts for Live Backend...")

joblib.dump(xgb_model, 'ser_xgb_model.joblib')
joblib.dump(lgb_model, 'ser_lgb_model.joblib')
joblib.dump(cb_model,  'ser_cb_model.joblib')
joblib.dump(scaler,    'ser_ensemble_scaler.joblib')
joblib.dump(encoder,   'ser_ensemble_encoder.joblib')

weights_dict = {
    "xgb_weight": float(best_weights[0]),
    "lgb_weight": float(best_weights[1]),
    "cb_weight":  float(best_weights[2])
}
with open('ensemble_weights.json', 'w') as f:
    json.dump(weights_dict, f, indent=4)

print("SUCCESS: All 3 Models, Scaler, Encoder, and Weights saved to disk!")
print("Ready for live deployment.")